## 📹 Video Introduction

Watch this video before starting the module:

<div style="padding:56.25% 0 0 0;position:relative;">
  <iframe src="https://player.vimeo.com/video/YOUR_VIDEO_ID" style="position:absolute;top:0;left:0;width:100%;height:100%;" frameborder="0" allow="autoplay; fullscreen; picture-in-picture" allowfullscreen></iframe>
</div>

---

## 🔧 TF Publishing Architecture

From Lesson 3, you created URDF files. Now we need to **publish TFs** for ROS applications.

### Key Nodes:
1. **robot_state_publisher**: Takes URDF + joint states → Publishes TFs
2. **joint_state_publisher**: Provides joint states (for development)

### Check Current Setup:
```bash
ros2 launch urdf_tutorial display.launch.py model:=/path/to/your_robot.urdf.xacro
```

Then check rqt_graph:

![](images/publishing%20tf.pdf-image-000.png)

**Key Takeaway:** The `/tf` topic is essential, published by `robot_state_publisher`.

---

## 🔍 Understanding Requirements

**robot_state_publisher needs 2 inputs:**

1. **URDF** as `robot_description` parameter
2. **Joint States** from `/joint_states` topic

![](images/publishing%20tf.pdf-image-001.png)

### Check Parameters:
```bash
ros2 param list /robot_state_publisher
ros2 param get /robot_state_publisher robot_description
```

**Note:** `joint_state_publisher` (GUI) publishes fake joint states for development.

---

## 🚀 Manual Node Startup

### Terminal 1: robot_state_publisher
```bash
ros2 run robot_state_publisher robot_state_publisher \
  --ros-args -p robot_description:="$(xacro /path/to/robot.urdf.xacro)"
```

### Terminal 2: joint_state_publisher_gui
```bash
ros2 run joint_state_publisher_gui joint_state_publisher_gui
```

**Result:** TFs published on `/tf` topic!

---

## 👁️ Visualizing with RViz

### Terminal 3: Start RViz
```bash
ros2 run rviz2 rviz2
```

**Initial State (with errors):

![](images/publishing%20tf.pdf-image-002.png)

### Configuration Steps:
1. **Fixed Frame**: Set to `base_footprint`
2. **Add RobotModel**: Topic → `/robot_description`
3. **Add TF Display**: Shows coordinate frames
4. **Adjust Alpha**: 0.8 for transparency

### Save Configuration:
```bash
File → Save Config As → urdf_config.rviz
```

![](images/publishing%20tf.pdf-image-003.png)

---

## 📦 Creating a ROS 2 Package

**Why package?** Organize, share, install properly.

### 1. Create New Workspace:
```bash
mkdir -p ~/my_robot_ws/src
```

### 2. Create Description Package:
```bash
cd ~/my_robot_ws/src
ros2 pkg create my_robot_description --build-type ament_cmake
cd my_robot_description
rm -r include src  # No C++ code
```

---

## 📁 Package Structure

```
my_robot_description/
├── CMakeLists.txt
├── package.xml
├── launch/
│   └── display.launch.xml
├── rviz/
│   └── urdf_config.rviz
└── urdf/
    ├── common_properties.xacro
    ├── mobile_base.xacro
    └── my_robot.urdf.xacro
```

### Install Folders in CMakeLists.txt:
```cmake
install(
  DIRECTORY urdf rviz launch
  DESTINATION share/${PROJECT_NAME}/
)
```

---

## 🚀 XML Launch File

**File:** `display.launch.xml`

```xml
<launch>
  <!-- Define paths -->
  <let name="urdf_path" 
       value="$(find-pkg-share my_robot_description)/urdf/my_robot.urdf.xacro" />
  <let name="rviz_config" 
       value="$(find-pkg-share my_robot_description)/rviz/urdf_config.rviz" />

  <!-- robot_state_publisher -->
  <node pkg="robot_state_publisher" exec="robot_state_publisher">
    <param name="robot_description" 
           value="$(command 'xacro $(var urdf_path)')" />
  </node>

  <!-- joint_state_publisher_gui -->
  <node pkg="joint_state_publisher_gui" 
        exec="joint_state_publisher_gui" />

  <!-- RViz -->
  <node pkg="rviz2" exec="rviz2" args="-d $(var rviz_config)" />
</launch>
```

### Launch It:
```bash
cd ~/my_robot_ws
colcon build --packages-select my_robot_description
source install/setup.bash
ros2 launch my_robot_description display.launch.xml
```

---

## 🧠 Challenge

In your launch file, change the RViz fixed frame from `base_footprint` to `base_link`.

Relaunch and observe:

1. What changes in the visualization?
2. Why does the robot appear at a different position relative to the grid?
3. Which frame would you use if you wanted the grid to stay fixed while the robot moves?

*Write your answers here:*

**1.**

**2.**

**3.**

# Summary

In this module, you learned:

✅ How robot_state_publisher converts URDF + joint states → TFs  

✅ How to start required nodes from terminal  

✅ How to create a proper ROS 2 description package  

✅ How to write XML launch files for TF publishing  

✅ How to configure and save RViz settings

---

> Your robot is now properly packaged. But it only exists visually — no physics, no gravity, no collisions. Next lesson: bring your robot into a physics simulator.